# Step 1 diagnostic: ML maths

This notebook finds out which parts of the fanout ML Maths course you can skip, which you should review, and which you need to learn in full. It covers the prerequisites and samples every module (sets and logic, linear systems, vector spaces, spectral methods, projection and PCA, matrix calculus, practical training code, cost estimates, probability and inference).

**How it works**
- 8 tasks. Each coding task has a `# YOUR CODE` cell and a `CHECK` cell that prints `PASS` or a specific failure.
- Every task ends with a written question. Answer it in the cell below it. I grade those by hand.
- The last cell prints a summary. The rubric under it maps results to skip / review / learn fully.

**Rules**
- Do not edit `CHECK` cells. Run them as they are.
- Each task lists forbidden calls. The checks scan your function's source and also block those functions while your code runs.
- Some tasks limit loop nesting (for example "no Python loops"). Comprehensions count as loops.
- No looking things up for the first attempt. If you need a reference to finish, say so in the written answer. That is useful information for the plan, not a penalty.
- Don't read the check cells for hints. Some contain the reference implementation.

**Runtime**: CPU only. Colab free tier or a laptop with `numpy` and `scipy`. Every check runs in under 10 seconds.

**Time**: about 6 to 8 hours in total. Stop a task after 75 minutes and write down where you got stuck.

When you finish, send me the notebook with outputs saved (File > Download > .ipynb).

In [ ]:
# Setup and check helpers: do not edit this cell
import numpy as np, scipy, scipy.linalg, scipy.stats, scipy.special
import ast, inspect, textwrap, contextlib, importlib, base64, json
np.set_printoptions(precision=4, suppress=True)
RESULTS = {}

class CheckFail(Exception):
    pass

def expect(cond, msg):
    if not cond:
        raise CheckFail(msg)

def _tree(fn):
    return ast.parse(textwrap.dedent(inspect.getsource(fn)))

def loop_depth(fn):
    def depth(node, d=0):
        best = d
        for ch in ast.iter_child_nodes(node):
            nd = d + isinstance(ch, (ast.For, ast.While, ast.comprehension))
            best = max(best, depth(ch, nd))
        return best
    return depth(_tree(fn))

def names_used(fn):
    out = set()
    for n in ast.walk(_tree(fn)):
        if isinstance(n, ast.Attribute):
            out.add(n.attr)
        elif isinstance(n, ast.Name):
            out.add(n.id)
        elif isinstance(n, ast.alias):
            out.add(n.name.split(".")[-1])
    return out

def static_rules(fns, banned, max_loops):
    for fn in fns:
        bad = names_used(fn) & set(banned)
        expect(not bad, f"{fn.__name__} uses forbidden name(s): {sorted(bad)}")
        if max_loops is not None:
            ld = loop_depth(fn)
            expect(ld <= max_loops,
                   f"{fn.__name__} has loop nesting depth {ld}; at most {max_loops} allowed (vectorise it)")

@contextlib.contextmanager
def blocked(*paths):
    saved = []
    for p in paths:
        mod, attr = p.rsplit(".", 1)
        m = importlib.import_module(mod)
        if not hasattr(m, attr):
            continue
        saved.append((m, attr, getattr(m, attr)))
        def boom(*a, _p=p, **k):
            raise CheckFail(f"your code called forbidden {_p}")
        setattr(m, attr, boom)
    try:
        yield
    finally:
        for m, a, f in saved:
            setattr(m, a, f)

def run_check(tid, body):
    try:
        body()
    except NotImplementedError:
        RESULTS[tid] = False
        print(f"FAIL {tid}: not implemented yet (NotImplementedError)")
    except CheckFail as e:
        RESULTS[tid] = False
        print(f"FAIL {tid}: {e}")
    except Exception as e:
        RESULTS[tid] = False
        print(f"FAIL {tid}: your code raised {type(e).__name__}: {e}")
    else:
        RESULTS[tid] = True
        print(f"PASS {tid}")
print("helpers loaded")

## Task 1: Masked argmax for a decoding step (modules 01, 02)

You are writing the token-selection step of a constrained decoder. `scores` has shape `(..., V)`. `mask` is a boolean array of the same shape: `True` means the token is allowed. Write `masked_argmax(scores, mask)` that returns, for every leading index, the position of the largest allowed score.

Spec:
- Returns an `int64` array of shape `scores.shape[:-1]`.
- NaN scores count as not allowed, even where `mask` is `True`.
- Ties go to the lowest index.
- `-inf` is a real score. If every allowed score in a row is `-inf`, return the first allowed index.
- If a row has no allowed entries (including `V == 0`), return `-1`.

Forbidden: `argmax`, `nanargmax`, `argmin`, `argsort`, `argpartition`, `sort`, `where` with one argument. No Python loops.

In [ ]:
def masked_argmax(scores, mask):
    # YOUR CODE
    raise NotImplementedError

In [ ]:
# CHECK: do not edit this cell
def _ref_t1(scores, mask):
    s = np.asarray(scores, float); m = np.asarray(mask, bool)
    lead = s.shape[:-1]; rows = int(np.prod(lead))
    flat_s = s.reshape(rows, s.shape[-1]); flat_m = m.reshape(rows, s.shape[-1])
    out = []
    for row_s, row_m in zip(flat_s, flat_m):
        best, bi = None, -1
        for i, (v, ok) in enumerate(zip(row_s, row_m)):
            if ok and not np.isnan(v) and (best is None or v > best):
                best, bi = v, i
        out.append(bi)
    return np.array(out, dtype=np.int64).reshape(lead)

def _t1():
    static_rules([masked_argmax], ["argmax", "nanargmax", "argmin", "argsort", "argpartition", "sort", "nonzero", "flatnonzero", "argwhere"], 0)
    rng = np.random.default_rng(1)
    cases = []
    s = rng.integers(0, 4, size=(6, 5, 7)).astype(float); m = rng.random((6, 5, 7)) < 0.6
    cases.append(("random with ties", s, m))
    s2 = rng.normal(size=(4, 6)); s2[0] = np.nan; s2[1, :3] = np.nan; m2 = np.ones((4, 6), bool)
    cases.append(("NaN scores", s2, m2))
    s3 = np.full((3, 5), -np.inf); m3 = np.array([[0,0,1,1,0],[0,0,0,0,0],[1,0,0,0,1]], bool)
    cases.append(("all -inf allowed", s3, m3))
    s4 = np.array([[1.0, np.inf, np.inf, 2.0]]); cases.append(("+inf ties", s4, np.ones((1, 4), bool)))
    cases.append(("V == 0", np.zeros((3, 0)), np.zeros((3, 0), bool)))
    cases.append(("no rows", np.zeros((0, 5)), np.zeros((0, 5), bool)))
    cases.append(("1-D input", np.array([3.0, 9.0, 9.0, 1.0]), np.array([1, 0, 1, 1], bool)))
    for name, s, m in cases:
        ref = _ref_t1(s, m)
        with blocked("numpy.argmax", "numpy.nanargmax", "numpy.argsort", "numpy.sort"):
            got = masked_argmax(s.copy(), m.copy())
        got = np.asarray(got)
        expect(got.shape == ref.shape, f"[{name}] shape {got.shape}, expected {ref.shape}")
        expect(got.dtype == np.int64, f"[{name}] dtype {got.dtype}, expected int64")
        expect(np.array_equal(got, ref), f"[{name}] got {got.ravel()[:8]}, expected {ref.ravel()[:8]}")
run_check("T1", _t1)

**Written question.** (a) Write the negation of "for every $\varepsilon > 0$ there is an $N$ such that for all $n \ge N$, $|a_n - L| < \varepsilon$", with no "not" in front of a quantifier.
(b) A colleague says: "If validation loss goes down every epoch, the model is not overfitting." Write the converse and the contrapositive. Which one is equivalent to the claim? Is the claim itself true? Give a concrete counterexample or argument.
(c) Logistic regression on linearly separable data: does $\arg\min_w L(w)$ exist? What is $\inf_w L(w)$? What does this mean for gradient descent without regularisation?

**Your answer:**

_(write here)_

## Task 2: A linear solver you can trust (module 03)

Write `gauss_solve(A, B)` that solves $AX = B$ by Gaussian elimination with partial pivoting.

Spec:
- `A` is `(n, n)`. `B` is `(n,)` or `(n, k)`. Return a float64 array with the same shape as `B`.
- Accept integer inputs. Do not modify `A` or `B` in place.
- Raise `np.linalg.LinAlgError` if `A` is singular to working precision (use a pivot threshold of `n * eps * max|A|`).
- Must be backward stable on ill-conditioned but non-singular inputs.

Forbidden: `solve`, `inv`, `lstsq`, `pinv`, `qr`, `lu`, `lu_factor`, `lu_solve`, `cho_solve`, `svd`, `solve_triangular`. At most one level of Python loop (a loop over columns is fine; a loop over rows inside it is not).

In [ ]:
def gauss_solve(A, B):
    # YOUR CODE
    raise NotImplementedError

In [ ]:
# CHECK: do not edit this cell
def _t2():
    static_rules([gauss_solve], ["solve", "inv", "lstsq", "pinv", "qr", "lu", "lu_factor", "lu_solve", "cho_solve", "svd", "solve_triangular"], 1)
    banned = ("numpy.linalg.solve", "numpy.linalg.inv", "numpy.linalg.lstsq", "scipy.linalg.solve", "scipy.linalg.lu_factor", "scipy.linalg.inv")
    rng = np.random.default_rng(2)
    A = rng.normal(size=(60, 60)); B = rng.normal(size=(60, 4))
    A0, B0 = A.copy(), B.copy()
    with blocked(*banned): X = gauss_solve(A, B)
    expect(np.array_equal(A, A0) and np.array_equal(B, B0), "your function modified A or B in place")
    expect(X.shape == (60, 4), f"shape {X.shape}, expected (60, 4)")
    expect(np.allclose(X, np.linalg.solve(A, B), rtol=1e-8, atol=1e-10), "random 60x60 system: answer differs from reference")
    b = rng.normal(size=60)
    with blocked(*banned): x = gauss_solve(A, b)
    expect(x.shape == (60,), f"1-D right-hand side should give shape (60,), got {x.shape}")
    try:
        with blocked(*banned): x = gauss_solve(np.array([[1e-20, 1.0], [1.0, 1.0]]), np.array([1.0, 2.0]))
    except np.linalg.LinAlgError:
        raise CheckFail("[[1e-20, 1], [1, 1]] is well conditioned but you raised LinAlgError (pivoting?)")
    expect(np.allclose(x, [1.0, 1.0], atol=1e-10), f"tiny-pivot system gives {x}, expected about [1, 1] (pivoting?)")
    with blocked(*banned): x = gauss_solve(np.array([[2, 1], [1, 3]]), np.array([3, 5]))
    expect(np.allclose(x, [0.8, 1.4]) and x.dtype == np.float64, "integer input: wrong answer or dtype")
    with blocked(*banned): x = gauss_solve(np.array([[4.0]]), np.array([2.0]))
    expect(np.allclose(x, [0.5]), "1x1 system wrong")
    for S in (np.array([[1.0, 2.0], [2.0, 4.0]]), np.zeros((3, 3)), np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]], float)):
        try:
            with blocked(*banned): gauss_solve(S, np.ones(len(S)))
        except np.linalg.LinAlgError:
            pass
        else:
            raise CheckFail(f"singular matrix {S.tolist()} did not raise LinAlgError")
    H = scipy.linalg.hilbert(9); xt = np.ones(9); bh = H @ xt
    with blocked(*banned): xh = gauss_solve(H, bh)
    rel_res = np.linalg.norm(H @ xh - bh) / (np.linalg.norm(H, 2) * np.linalg.norm(xh))
    expect(rel_res < 1e-14, f"Hilbert(9): relative residual {rel_res:.1e} too large; not backward stable")
run_check("T2", _t2)

**Written question.** (a) In the `[[1e-20, 1], [1, 1]]` case, what answer does elimination without pivoting give in float64, and why?
(b) For the Hilbert(9) system, the residual is tiny but `x` is visibly far from all ones. Explain using the condition number. Roughly how many correct digits should you expect when $\kappa(H) \approx 5 \times 10^{11}$?
(c) On call: a teammate fits linear regression with `np.linalg.solve(X.T @ X, X.T @ y)`. $\kappa(X) = 10^8$. The coefficients look random. What is $\kappa(X^\top X)$, what is going wrong, and what would you change in the code?

**Your answer:**

_(write here)_

## Task 3: Rank and null space by row reduction (module 04)

Write `rank_nullspace(A, tol=None)` returning `(r, N)`:
- `r`: the rank of `A`.
- `N`: an `(n, n - r)` array whose columns are a basis of the null space of `A` (they need not be orthonormal). If `r == n`, return shape `(n, 0)`.
- Treat a pivot as zero if its absolute value is at most `tol`. Default `tol = 1e-9 * max|A|`.

Use row reduction to reduced row echelon form. Forbidden: `matrix_rank`, `svd`, `null_space`, `orth`, `qr`, `lstsq`, `pinv`, `eig`, `eigh`, `eigvals`, `eigvalsh`, `solve`. At most one level of Python loop.

In [ ]:
def rank_nullspace(A, tol=None):
    # YOUR CODE
    raise NotImplementedError

In [ ]:
# CHECK: do not edit this cell
def _t3():
    static_rules([rank_nullspace], ["matrix_rank", "svd", "null_space", "orth", "qr", "lstsq", "pinv", "eig", "eigh", "eigvals", "eigvalsh", "solve"], 1)
    banned = ("numpy.linalg.matrix_rank", "numpy.linalg.svd", "scipy.linalg.null_space", "scipy.linalg.svd", "numpy.linalg.qr")
    rng = np.random.default_rng(3)
    cases = [
        ("rank-3 6x8", rng.normal(size=(6, 3)) @ rng.normal(size=(3, 8)), 3),
        ("zero 3x4", np.zeros((3, 4)), 0),
        ("full column rank 5x3", rng.normal(size=(5, 3)), 3),
        ("wide full row rank 3x7", rng.normal(size=(3, 7)), 3),
        ("rank-2 plus 1e-13 noise", rng.normal(size=(7, 2)) @ rng.normal(size=(2, 6)) + 1e-13 * rng.normal(size=(7, 6)), 2),
        ("duplicate columns", np.repeat(rng.normal(size=(4, 2)), 3, axis=1), 2),
        ("integer matrix", np.array([[1, 2, 3], [2, 4, 6], [1, 0, 1]]), 2),
    ]
    for name, A, r_true in cases:
        A0 = np.array(A, copy=True)
        with blocked(*banned): r, N = rank_nullspace(A)
        n = A0.shape[1]
        expect(r == r_true, f"[{name}] rank {r}, expected {r_true}")
        N = np.asarray(N, float)
        expect(N.shape == (n, n - r_true), f"[{name}] N has shape {N.shape}, expected {(n, n - r_true)}")
        if N.shape[1]:
            scale = max(np.abs(A0).max(), 1.0) * max(np.abs(N).max(), 1.0)
            expect(np.abs(A0 @ N).max() <= 1e-8 * scale, f"[{name}] A @ N is not close to zero (max {np.abs(A0 @ N).max():.1e})")
            expect(np.linalg.matrix_rank(N) == n - r_true, f"[{name}] columns of N are not independent")
run_check("T3", _t3)

**Written question.** (a) An embedding table $E$ is $50{,}000 \times 768$. Its singular values drop from about 10 to below $10^{-6}$ after the 120th. What are the numerical rank of $E$ and the dimension of the null space of $E$ (as a map $\mathbb{R}^{768} \to \mathbb{R}^{50000}$)? What does that say about the embeddings?
(b) Why can't you get the rank reliably from row reduction with `tol=0` on real data? What made the "1e-13 noise" case pass or fail for you?
(c) Rank-nullity: a linear layer maps $\mathbb{R}^{4096} \to \mathbb{R}^{1024}$. What is the smallest possible dimension of its null space? Give one practical consequence for what the next layer can recover.

**Your answer:**

_(write here)_

## Task 4: PCA from scratch (modules 05 to 08)

Write `pca(X, k)` for a data matrix `X` of shape `(n, d)` (rows are samples). Return `(components, explained_variance, Z)`:
- `components`: `(k, d)`, orthonormal rows, ordered by decreasing variance.
- `explained_variance`: `(k,)`, the variance along each component, using the sample covariance (divide by `n - 1`).
- `Z`: `(n, k)`, the scores (projections of the centred data).
- Sign convention: in each component, the entry with the largest absolute value is positive.
- Raise `ValueError` unless `1 <= k <= min(n, d)`.

Forbidden: `svd`, `PCA`, `sklearn`, `TruncatedSVD`. You may use `np.linalg.eigh`. No Python loops.

In [ ]:
def pca(X, k):
    # YOUR CODE
    raise NotImplementedError

In [ ]:
# CHECK: do not edit this cell
def _ref_pca(X, k):
    X = np.asarray(X, float); Xc = X - X.mean(0)
    U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
    V = Vt[:k]; s = np.sign(V[np.arange(k), np.abs(V).argmax(1)]); V = V * s[:, None]
    return V, S[:k] ** 2 / (len(X) - 1), Xc @ V.T

def _t4():
    static_rules([pca], ["svd", "PCA", "sklearn", "TruncatedSVD"], 0)
    banned = ("numpy.linalg.svd", "scipy.linalg.svd")
    rng = np.random.default_rng(4)
    def make(n, d, offset):
        Q, _ = np.linalg.qr(rng.normal(size=(d, d)))
        return rng.normal(size=(n, d)) * np.linspace(6, 0.5, d) @ Q.T + offset
    cases = [("n=500 d=10 k=3, mean offset 1e3", make(500, 10, 1e3), 3),
             ("d > n: n=20 d=50 k=5", make(20, 50, 0.0), 5),
             ("k = d", make(300, 6, -7.0), 6),
             ("float32 input", make(400, 8, 2.0).astype(np.float32), 4)]
    for name, X, k in cases:
        Vr, er, Zr = _ref_pca(X.astype(float), k)
        X0 = X.copy()
        with blocked(*banned): V, ev, Z = pca(X, k)
        expect(np.array_equal(X, X0), f"[{name}] X was modified in place")
        V, ev, Z = map(np.asarray, (V, ev, Z))
        expect(V.shape == (k, X.shape[1]) and ev.shape == (k,) and Z.shape == (X.shape[0], k), f"[{name}] wrong output shapes {V.shape}, {ev.shape}, {Z.shape}")
        expect(np.allclose(V @ V.T, np.eye(k), atol=1e-6), f"[{name}] components are not orthonormal")
        tol = 1e-3 if X.dtype == np.float32 else 1e-6
        expect(np.allclose(ev, er, rtol=tol), f"[{name}] explained variance {ev[:3]} vs reference {er[:3]} (n-1? centring?)")
        expect(np.allclose(V, Vr, atol=tol * 10), f"[{name}] components differ from reference (order, sign convention, centring?)")
        expect(np.allclose(Z, Zr, atol=tol * 100), f"[{name}] scores Z differ from reference")
    for bad_k in (0, 11, -1):
        try:
            pca(make(30, 10, 0.0), bad_k)
        except ValueError:
            pass
        else:
            raise CheckFail(f"k={bad_k} with d=10 should raise ValueError")
run_check("T4", _t4)

**Written question.** (a) Libraries compute PCA from the SVD of the centred $X$, not from `eigh` of $X^\top X$. Explain why in terms of condition numbers, and give a case where your `eigh` version loses most of its accuracy on the smallest components.
(b) You run PCA on raw features where one column is income in dollars and the rest are in $[0, 1]$. PC1 explains 99.9% of the variance. What happened, and what should you have done?
(c) Show that the mean squared reconstruction error from keeping $k$ components equals the sum of the discarded eigenvalues (up to the $n$ vs $n-1$ factor). Two or three lines are enough.

**Your answer:**

_(write here)_

## Task 5: Softmax cross-entropy layer with a hand-derived backward (module 09)

A linear classifier computes logits $Z = XW + b$ for $X \in \mathbb{R}^{N \times D}$, $W \in \mathbb{R}^{D \times C}$, $b \in \mathbb{R}^{C}$. Write `xent_fwd_bwd(X, W, b, y, ignore_index=-100)` that returns `(loss, dW, db)`.

Spec:
- `loss` is the mean negative log-likelihood over rows where `y != ignore_index`.
- If every row is ignored, return `loss = 0.0` and zero gradients.
- Must stay finite for logits around $\pm 10^4$.
- Derive the gradient by hand. The check compares it to finite differences of a reference loss.

Forbidden: `logsumexp`, `softmax`, `log_softmax`, `torch`, `jax`, `autograd`, `grad`. No Python loops.

In [ ]:
def xent_fwd_bwd(X, W, b, y, ignore_index=-100):
    # YOUR CODE
    raise NotImplementedError

In [ ]:
# CHECK: do not edit this cell
def _ref_xent(X, W, b, y, ig=-100):
    Z = X @ W + b; v = y != ig
    if not v.any(): return 0.0
    lp = Z - scipy.special.logsumexp(Z, axis=1, keepdims=True)
    return -lp[np.arange(len(y))[v], y[v]].mean()

def _t5():
    static_rules([xent_fwd_bwd], ["logsumexp", "softmax", "log_softmax", "torch", "jax", "autograd", "grad"], 0)
    banned = ("scipy.special.logsumexp", "scipy.special.softmax", "scipy.special.log_softmax")
    rng = np.random.default_rng(5)
    N, D, C = 7, 5, 4
    X = rng.normal(size=(N, D)); W = rng.normal(size=(D, C)); b = rng.normal(size=C)
    y = rng.integers(0, C, size=N); y[[1, 4]] = -100
    with blocked(*banned): loss, dW, db = xent_fwd_bwd(X, W, b, y)
    expect(np.shape(dW) == (D, C) and np.shape(db) == (C,), f"gradient shapes {np.shape(dW)}, {np.shape(db)}")
    expect(abs(loss - _ref_xent(X, W, b, y)) < 1e-10, f"loss {loss} vs reference {_ref_xent(X, W, b, y)} (mean over non-ignored rows?)")
    h = 1e-6
    for name, P, G in (("W", W, dW), ("b", b, db)):
        num = np.zeros_like(P)
        for idx in np.ndindex(P.shape):
            Pp = P.copy(); Pp[idx] += h; Pm = P.copy(); Pm[idx] -= h
            args_p = (X, Pp, b, y) if name == "W" else (X, W, Pp, y)
            args_m = (X, Pm, b, y) if name == "W" else (X, W, Pm, y)
            num[idx] = (_ref_xent(*args_p) - _ref_xent(*args_m)) / (2 * h)
        err = np.abs(num - G).max() / max(np.abs(num).max(), 1e-12)
        expect(err < 1e-6, f"d{name} disagrees with finite differences (relative error {err:.1e})")
    Xb = X * 3e3
    with blocked(*banned): loss, dW, db = xent_fwd_bwd(Xb, W, b, y)
    expect(np.isfinite(loss) and np.all(np.isfinite(dW)) and np.all(np.isfinite(db)), "non-finite output for large logits (stability?)")
    expect(abs(loss - _ref_xent(Xb, W, b, y)) < 1e-6 * max(1, abs(loss)), "loss wrong for large logits")
    y_all = np.full(N, -100)
    with blocked(*banned): loss, dW, db = xent_fwd_bwd(X, W, b, y_all)
    expect(loss == 0.0 and not np.any(dW) and not np.any(db), "all rows ignored should give loss 0 and zero gradients")
    y_one = np.full(N, -100); y_one[3] = 2
    with blocked(*banned): l1, dW1, db1 = xent_fwd_bwd(X, W, b, y_one)
    l2, dW2, db2 = xent_fwd_bwd(X[3:4], W, b, y_one[3:4])
    expect(abs(l1 - l2) < 1e-12 and np.allclose(dW1, dW2) and np.allclose(db1, db2), "ignored rows still affect the result")
    with blocked(*banned): l32, _, _ = xent_fwd_bwd(X.astype(np.float32), W.astype(np.float32), b.astype(np.float32), y)
    expect(abs(l32 - _ref_xent(X, W, b, y)) < 1e-4, "float32 inputs give a wrong loss")
run_check("T5", _t5)

**Written question.** (a) Write out $\partial L / \partial Z$ for one non-ignored row, and show how $dW$ and $db$ follow from it with the chain rule. State the shapes at each step.
(b) On call: a model trained in fp16 gives NaN loss at step 3,000. Logged max logit is 70. The softmax is written as `exp(z) / exp(z).sum()`. Above which logit value does `exp` overflow in fp16 (max finite value 65504)? Which line produces the NaN, and what are two fixes?
(c) Your loss uses the mean over non-ignored rows. A teammate divides by the batch size $N$ instead. With 30% padding tokens, what does that do to the effective learning rate, and why can it change between batches?

**Your answer:**

_(write here)_

## Task 6: Find the bugs in a training script (modules 08, 09)

The cell below trains an L2-regularised logistic regression with early stopping. It runs without errors and prints plausible numbers. It contains **8 planted bugs**. Find and fix all of them in place. Keep the function names and signatures.

Intended behaviour:
- `make_split` shuffles rows (keeping X and y aligned) and holds out `val_frac` of them.
- `prepare` standardises features using training statistics only.
- `loss_and_grad` returns the mean log-loss plus $\frac{\lambda}{2}\lVert w\rVert^2$ (the bias is not regularised), and its exact gradient.
- `train` runs gradient descent, keeps the weights with the lowest validation loss, and stops after `patience` epochs in a row without improvement.
- `predict` returns 0/1 labels; `accuracy` returns the fraction correct.

The check tests each function separately and lists every probe that still fails.

In [ ]:
def sigmoid(z):
    return 0.5 * (1.0 + np.tanh(0.5 * z))

def make_split(X, y, val_frac=0.25, seed=0):
    rng = np.random.default_rng(seed)
    X = X[rng.permutation(len(X))]
    y = y[rng.permutation(len(y))]
    n_val = int(len(X) * val_frac)
    return X[n_val:], y[n_val:], X[:n_val], y[:n_val]

def prepare(X_train, X_val):
    X_all = np.vstack([X_train, X_val])
    mu, sd = X_all.mean(axis=0), X_all.std(axis=0)
    sd = np.where(sd == 0, 1.0, sd)
    return (X_train - mu) / sd, (X_val - mu) / sd

def loss_and_grad(w, b, X, y, lam):
    z = X @ w + b
    loss = np.mean(np.logaddexp(0.0, z) - y * z) + 0.5 * lam * np.sum(w ** 2)
    r = sigmoid(z) - y
    dw = X.T @ r + lam * w
    db = np.mean(r) + lam * b
    return loss, dw, db

def train(X_tr, y_tr, X_va, y_va, lr=0.1, lam=1e-2, epochs=500, patience=20):
    w = np.zeros(X_tr.shape[1])
    b = 0.0
    best_val, best_w, best_b = np.inf, w, b
    bad, hist = 0, []
    for ep in range(epochs):
        _, dw, db = loss_and_grad(w, b, X_tr, y_tr, lam)
        w -= lr * dw
        b -= lr * db
        val, _, _ = loss_and_grad(w, b, X_va, y_va, 0.0)
        hist.append(val)
        if val < best_val:
            best_val, best_w, best_b = val, w, b
        else:
            bad += 1
            if bad >= patience:
                break
    return best_w, best_b, hist

def predict(w, b, X):
    return (X @ w + b > 0.5).astype(int)

def accuracy(w, b, X, y):
    pred = predict(w, b, X).reshape(-1, 1)
    return float(np.mean(pred == y))

# demo run
_rng = np.random.default_rng(0)
_X = _rng.normal(size=(1000, 5)) * [1, 10, 0.1, 1, 3] + [0, 50, 0, 0, 0]
_y = (_X @ np.array([1.0, 0.1, 5.0, -1.0, 0.0]) - 5 + _rng.normal(size=1000) > 0).astype(float)
Xtr, ytr, Xva, yva = make_split(_X, _y)
Xtr, Xva = prepare(Xtr, Xva)
w_hat, b_hat, hist = train(Xtr, ytr, Xva, yva)
print(f"epochs run: {len(hist)}, best val loss: {min(hist):.4f}, val accuracy: {accuracy(w_hat, b_hat, Xva, yva):.3f}")

In [ ]:
# CHECK: do not edit this cell
def _t6():
    fails = []
    def probe(name, fn):
        try:
            ok, msg = fn()
        except Exception as e:
            ok, msg = False, f"raised {type(e).__name__}: {e}"
        if not ok:
            fails.append(f"{name}: {msg}")
    rng = np.random.default_rng(6)

    def p_split():
        X = np.arange(40.0)[:, None] * np.ones((1, 3)); y = (np.arange(40) % 3 == 0).astype(float)
        a, ya, c, yc = make_split(X, y, 0.25, seed=1)
        ok = np.array_equal(ya, (a[:, 0] % 3 == 0)) and np.array_equal(yc, (c[:, 0] % 3 == 0)) and len(c) == 10
        return ok, "X and y rows no longer line up after the split"
    probe("make_split", p_split)

    def p_prep():
        A = rng.normal(3, 2, size=(50, 4)); B = rng.normal(-5, 7, size=(20, 4))
        a, b_ = prepare(A, B)
        mu, sd = A.mean(0), A.std(0)
        return np.allclose(a, (A - mu) / sd) and np.allclose(b_, (B - mu) / sd), "scaling statistics are not from the training set only"
    probe("prepare", p_prep)

    X = rng.normal(size=(30, 4)); y = (rng.random(30) < 0.4).astype(float)
    w0 = rng.normal(size=4); b0 = 0.7; lam = 0.5
    def ref_loss(w, b):
        z = X @ w + b
        return np.mean(np.logaddexp(0.0, z) - y * z) + 0.5 * lam * np.sum(w ** 2)
    def p_loss():
        L, _, _ = loss_and_grad(w0, b0, X, y, lam)
        return abs(float(L) - ref_loss(w0, b0)) < 1e-10, "loss value is wrong"
    probe("loss_and_grad loss", p_loss)
    def p_dw():
        _, dw, _ = loss_and_grad(w0, b0, X, y, lam)
        num = np.array([(ref_loss(w0 + 1e-6 * e, b0) - ref_loss(w0 - 1e-6 * e, b0)) / 2e-6 for e in np.eye(4)])
        return np.shape(dw) == (4,) and np.allclose(dw, num, rtol=1e-5, atol=1e-7), "dw does not match finite differences of the loss"
    probe("loss_and_grad dw", p_dw)
    def p_db():
        _, _, db = loss_and_grad(w0, b0, X, y, lam)
        num = (ref_loss(w0, b0 + 1e-6) - ref_loss(w0, b0 - 1e-6)) / 2e-6
        return abs(float(db) - num) < 1e-6, "db does not match finite differences of the loss"
    probe("loss_and_grad db", p_db)

    def scripted(vals):
        it = iter(vals)
        def fake(w, b, Xa, ya, lam_):
            if Xa is Xv:
                return next(it), None, None
            return 0.0, np.ones_like(w), 1.0
        return fake
    Xt, Xv = np.zeros((5, 3)), np.zeros((4, 3))
    vals = [5, 4, 6, 6, 3, 6, 6, 6, 6, 6, 6, 6]
    real = globals()["loss_and_grad"]
    def p_best():
        globals()["loss_and_grad"] = scripted(vals)
        try:
            bw, bb, h = train(Xt, np.zeros(5), Xv, np.zeros(4), lr=0.1, lam=0.0, epochs=12, patience=3)
        finally:
            globals()["loss_and_grad"] = real
        return np.allclose(bw, -0.5) and abs(bb + 0.5) < 1e-12, "returned weights are not the ones from the best validation epoch"
    probe("train best weights", p_best)
    def p_pat():
        globals()["loss_and_grad"] = scripted(vals)
        try:
            _, _, h = train(Xt, np.zeros(5), Xv, np.zeros(4), lr=0.1, lam=0.0, epochs=12, patience=3)
        finally:
            globals()["loss_and_grad"] = real
        return len(h) == 8, f"early stopping fired after {len(h)} epochs; expected 8 for this validation curve"
    probe("train early stopping", p_pat)

    def p_pred():
        Xp = np.array([[0.2], [0.4], [-0.1], [3.0]])
        p = np.asarray(predict(np.array([1.0]), 0.0, Xp))
        return np.array_equal(p.ravel(), [1, 1, 0, 1]) and p.shape == (4,), f"predict gives {p.ravel()} for logits [0.2, 0.4, -0.1, 3.0]"
    probe("predict", p_pred)
    def p_acc():
        Xp = np.array([[1.0], [-1.0], [2.0], [-3.0], [0.5]]); yp = np.array([1.0, 0.0, 0.0, 0.0, 1.0])
        a = accuracy(np.array([1.0]), 0.0, Xp, yp)
        return abs(a - 0.8) < 1e-12, f"accuracy {a}, expected 0.8"
    probe("accuracy", p_acc)

    expect(not fails, f"{len(fails)} of 9 probes fail:\n  - " + "\n  - ".join(fails))
run_check("T6", _t6)

**Written question.** For each of the 8 bugs: one line on what it was, and one line on the symptom you would see in metrics or logs if it shipped. Then: which of the bugs would not show up in the training-loss curve at all, and how would you catch those in a real codebase (tests, assertions, monitoring)?

**Your answer:**

_(write here)_

## Task 7: Estimates on paper (cost reasoning, modules 09 to 12)

Fill in `ANSWERS` with numbers. Use 1 GB = $10^9$ bytes. Count a multiply-add as 2 FLOPs. Show your working in the written cell below; the check only looks at the numbers (tolerance 10 to 30%).

- **a.** Memory in GB of a float32 matrix with $n = 2{,}000{,}000$ rows and $d = 1024$ columns.
- **b.** FLOPs to form $X^\top X$ for that matrix (no symmetry tricks).
- **c.** Seconds for (b) on a T4 GPU in fp32 (peak 8.1 TFLOP/s), assuming 50% of peak.
- **d.** Memory in GB to store the full Hessian of a model with $10^6$ parameters in float32.
- **e.** FLOPs for one training step (forward plus backward) of an MLP $784 \to 1024 \to 1024 \to 10$ at batch size 256. Ignore biases and activations; assume backward costs 2x forward.
- **f.** GB of memory for weights, gradients and optimiser state when training a 7B-parameter model with Adam in mixed precision: bf16 weights and gradients, plus fp32 master weights and fp32 Adam moments. Ignore activations.
- **g.** Monte Carlo samples needed to estimate a probability near $p = 0.01$ with standard error $0.001$.
- **h.** An alert fires for 99% of real incidents and for 1% of non-incidents. 0.1% of time windows contain a real incident. Probability that a given alert is a real incident?

In [ ]:
ANSWERS = {
    "a": None,  # GB
    "b": None,  # FLOPs
    "c": None,  # seconds
    "d": None,  # GB
    "e": None,  # FLOPs
    "f": None,  # GB
    "g": None,  # samples
    "h": None,  # probability
}

In [ ]:
# CHECK: do not edit this cell
_KEY = json.loads(base64.b64decode("eyJBIjogeyJhIjogOC4xOTIsICJiIjogNDE5NDAwMDAwMDAwMC4wLCAiYyI6IDEuMDM2LCAiZCI6IDQwMDAuMCwgImUiOiAyODYwMDAwMDAwLjAsICJmIjogMTEyLjAsICJnIjogOTkwMC4wLCAiaCI6IDAuMDkwMn0sICJUIjogeyJhIjogMC4xLCAiYiI6IDAuMTUsICJjIjogMC4zLCAiZCI6IDAuMSwgImUiOiAwLjE1LCAiZiI6IDAuMSwgImciOiAwLjEsICJoIjogMC4xfX0="))
def _t7():
    missing = [k for k, v in ANSWERS.items() if v is None]
    if len(missing) == len(_KEY["A"]):
        raise NotImplementedError
    expect(not missing, f"missing answers: {missing}")
    wrong = []
    for k, ref in _KEY["A"].items():
        v = float(ANSWERS[k])
        if not abs(v - ref) <= _KEY["T"][k] * abs(ref):
            wrong.append(k)
    expect(not wrong, f"outside tolerance: {wrong}. Recheck units and factors of 2.")
run_check("T7", _t7)

**Written question.** Show the working for a to h (one line each). Then: if you switch (a) to (c) to bf16, which numbers change and by how much? For (h), what single change to the alert would raise the answer the most: halving the miss rate or halving the false-positive rate? Why?

**Your answer:**

_(write here)_

## Task 8: Gaussian log-likelihood that survives high dimensions (modules 10 to 12)

Write `mvn_logpdf(X, mu, Sigma)` returning the log-density of $\mathcal{N}(\mu, \Sigma)$ at each row of `X` (shape `(N, d)`), as an array of shape `(N,)`.

Spec:
- Use a Cholesky factorisation. Raise `np.linalg.LinAlgError` if `Sigma` is not positive definite.
- Accurate for $d = 300$ with small variances, where $\det\Sigma$ underflows to 0.
- `N == 0` returns an empty array.

Allowed: `np.linalg.cholesky`, `scipy.linalg.solve_triangular`. Forbidden: `multivariate_normal`, `inv`, `det`, `slogdet`, `pinv`, `solve`, `lstsq`, `eigh`, `eig`. No Python loops.

In [ ]:
def mvn_logpdf(X, mu, Sigma):
    # YOUR CODE
    raise NotImplementedError

In [ ]:
# CHECK: do not edit this cell
def _t8():
    static_rules([mvn_logpdf], ["multivariate_normal", "inv", "det", "slogdet", "pinv", "solve", "lstsq", "eigh", "eig"], 0)
    banned = ("numpy.linalg.inv", "numpy.linalg.det", "numpy.linalg.slogdet", "numpy.linalg.solve", "numpy.linalg.eigh", "scipy.linalg.inv", "scipy.linalg.det")
    rng = np.random.default_rng(8)
    def spd(d, scale):
        A = rng.normal(size=(d, d)); return scale * (A @ A.T / d + 0.5 * np.eye(d))
    for name, d, scale, N in (("d=4", 4, 1.0, 50), ("d=1", 1, 2.0, 10), ("d=300 small variances", 300, 1e-3, 20)):
        S = spd(d, scale); mu = rng.normal(size=d); X = mu + rng.normal(size=(N, d)) @ np.linalg.cholesky(S).T
        ref = scipy.stats.multivariate_normal(mu, S).logpdf(X).reshape(N)
        with blocked(*banned): got = np.asarray(mvn_logpdf(X, mu, S))
        expect(got.shape == (N,), f"[{name}] shape {got.shape}, expected {(N,)}")
        expect(np.all(np.isfinite(got)), f"[{name}] non-finite values (det underflow?)")
        expect(np.allclose(got, ref, rtol=1e-8, atol=1e-6), f"[{name}] max abs error {np.abs(got - ref).max():.2e} vs scipy")
    with blocked(*banned): e = np.asarray(mvn_logpdf(np.zeros((0, 3)), np.zeros(3), np.eye(3)))
    expect(e.shape == (0,), f"N=0 should return shape (0,), got {e.shape}")
    try:
        with blocked(*banned): mvn_logpdf(np.zeros((2, 2)), np.zeros(2), np.array([[1.0, 2.0], [2.0, 1.0]]))
    except np.linalg.LinAlgError:
        pass
    else:
        raise CheckFail("indefinite Sigma did not raise LinAlgError")
run_check("T8", _t8)

**Written question.** (a) For $d = 300$ and $\Sigma = 10^{-3} I$, what is $\det \Sigma$, and what does float64 store for it? What is the log-determinant?
(b) Derive the MLE of $\mu$ and $\Sigma$ for i.i.d. Gaussian data (state the result, sketch the derivative step). Is $\hat\Sigma_{\text{MLE}}$ unbiased? By what factor is it off for $n = 5$?
(c) You fit a Gaussian to 64-dimensional features with $n = 40$ samples and call `mvn_logpdf` on new points. What happens, and give two fixes.
(d) A 95% confidence interval for a model's accuracy on 400 test examples is reported as $0.82 \pm 0.04$. Check this number, and say what "95%" does and does not mean here.

**Your answer:**

_(write here)_

## Summary

Run the cell below after all checks. It prints which coding tasks passed.

In [ ]:
# Summary: do not edit this cell
for t in ["T1", "T2", "T3", "T4", "T5", "T6", "T7", "T8"]:
    print(t, "PASS" if RESULTS.get(t) else "FAIL / not run")

## Rubric

I grade the written answers. The table shows how results map to the plan.

| Task | Fanout modules | Skip | Review | Learn fully |
|---|---|---|---|---|
| T1 masked argmax, logic | 01 Sets, 02 Logic | PASS, and (a) to (c) correct | PASS, one written part shaky | FAIL, or two or more written parts wrong |
| T2 Gaussian elimination | 03 Linear systems | PASS, and conditioning answers (b), (c) correct | PASS but (b) or (c) wrong | FAIL |
| T3 rank and null space | 04 Spaces | PASS, and rank-nullity reasoning correct | PASS, written answers shaky | FAIL |
| T4 PCA | 05 Spectral, 06 Projection, 07 Low rank, 08 Applications | PASS, and (a) to (c) correct | PASS, (a) or (c) wrong | FAIL |
| T5 cross-entropy backward | 09 Calculus (gradients, Jacobians, matrix calculus) | PASS, and the derivation in (a) is clean | PASS but needed references for the gradient | FAIL |
| T6 bug hunt | 08 Applications, 09 Optimisation | all 9 probes pass, and symptoms explained | 6 or 7 bugs fixed | 5 or fewer bugs fixed |
| T7 estimates | cost reasoning, 10 Probability | PASS, working shown | 6 or 7 of 8 correct | 5 or fewer correct |
| T8 Gaussian log-likelihood | 10 Probability, 11 Joint RVs, 12 Inference | PASS, and (b) to (d) correct | PASS, written parts shaky | FAIL |

Notes:
- Modules 09.06 to 09.07 (integration) and 10.05 to 10.06 (named distributions) are only probed indirectly. They default to "review" unless T7 and T8 go badly.
- "Skip" still means skimming the free fanout lesson and doing the practice sheet's hardest exercise. Since you chose full scope, nothing is dropped; the result decides depth and pace.
- If two or more of T2 to T5 fail, I add a Step 0 refresher before the main plan.